In [3]:
import anthropic
from dotenv import load_dotenv
import requests
import os
import json
from datetime import datetime
load_dotenv()
client = anthropic.Anthropic()
TRANSACTIONS_FILE = "transactions.json"
riot_key = os.getenv("RIOT_API_KEY")
messages=[]


tools=[
      {"name":"find_riot_account","description":"Find account in the RIOT games API by tagline and game name",        
        "input_schema":{
        "type": "object",
        "properties": {
            "server":{"type":"string","description":"The server this player is on, it is either americas, europe or asia, do not use like euw1 or kr"},
            "game_name": {"type": "string", "description": "The username/gamename for the account you are looking for"},
            "tag_line": {"type": "string", "description": "The tagnline for the account you are looking for"}

        },
            "required":["server","game_name","tag_line"]
        }},
        {"name":"find_masteries","description":"Find the top champions in masteries by account id",        
        "input_schema":{
        "type": "object",
        "properties": {
            "server":{"type":"string","description":"The server this player is on, this one it is set up with like euw1 or kr, do not use like europe"},
            "encryptedPUUID": {"type": "string", "description": "The id of the player that we want the masteries of"},

        },
            "required":["server","encryptedPUUID"]
        }},
            {"name":"find_match_history","description":"Find the match history of an account",        
        "input_schema":{
        "type": "object",
        "properties": {
            "server":{"type":"string","description":"The server this player is on, this one it is set up with like europe or asia"},
            "puuid": {"type": "string", "description": "The id of the player that we want the match history of"},

        },
            "required":["server","encryptedPUUID"]
        }},
            {"name":"find_match_by_id","description":"Find the top champions in masteries by account id",        
        "input_schema":{
        "type": "object",
        "properties": {
            "server":{"type":"string","description":"The server this player is on, this one it is set up with with like europe or asia"},
            "match_id": {"type": "string", "description": "The id of the match you want"},

        },
            "required":["server","match_id"]
        }}
]

def find_riot_account(server, game_name, tag_line):
    url = f"https://{server}.api.riotgames.com/riot/account/v1/accounts/by-riot-id/{game_name}/{tag_line}"
    headers = {"X-Riot-Token": riot_key}
    response = requests.get(url, headers=headers)
    if response.status_code == 200:
        return response.json()
    else:
        return f"Request failed with status {response.status_code}"


def find_masteries(server,encryptedPUUID):
    url = f"https://{server}.api.riotgames.com/lol/champion-mastery/v4/champion-masteries/by-puuid/{encryptedPUUID}/top/"
    headers = {"X-Riot-Token": riot_key}
    response = requests.get(url, headers=headers)
    top_ten_masteries = []
    if response.status_code == 200:
        return response.json()
    else:
        return f"Request failed with status {response.status_code}"


def find_match_history(server,puuid):
    url = f"https://{server}.api.riotgames.com/lol/match/v5/matches/by-puuid/{puuid}/ids/"
    headers = {"X-Riot-Token": riot_key}
    response = requests.get(url, headers=headers)
    top_ten_masteries = []
    if response.status_code == 200:
        return response.json()
    else:
        return f"Request failed with status {response.status_code}"

def find_match_by_id(server,match_id):
    url = f"https://{server}.api.riotgames.com/lol/match/v5/matches/{match_id}/"
    headers = {"X-Riot-Token": riot_key}
    response = requests.get(url, headers=headers)
    top_ten_masteries = []
    print(response.json()["info"]["queueId"])
    print(response.json()["info"]["gameMode"])
    if response.status_code == 200:
        return response.json()
    else:
        return f"Request failed with status {response.status_code}"


tool_functions = {"find_riot_account":find_riot_account,"find_masteries":find_masteries,"find_match_history":find_match_history,"find_match_by_id":find_match_by_id}
while True:
    user_text = input()
    if user_text == "exit":
        break
    messages.append({"role":"user","content":user_text})
    message = client.messages.create(
        tools=tools,
        model="claude-haiku-4-5-20251001",
        max_tokens=1024,
        messages=messages
    )
    messages.append({"role":"assistant","content":message.content})
    while message.stop_reason == "tool_use":
        tool_results = []
        for i in message.content:
            if i.type == "tool_use":
                result = tool_functions[i.name](**i.input)
                tool_results.append({
                    "type":"tool_result",
                    "tool_use_id":i.id,
                    "content":json.dumps(result)
                })
        messages.append({"role":"user","content":tool_results})
        message = client.messages.create(
            tools=tools,
            model="claude-haiku-4-5-20251001",
            max_tokens=1024,
            messages=messages
        )
        messages.append({"role": "assistant", "content": message.content})
        
    for block in message.content:
        if block.type == "text":
            print(block.text)


   Give me the latest match of the account Jovial Kar with the tag EUW


1700
CHERRY
## Latest Match Details for Jovial Kar

Here's the information about the latest match of Jovial Kar (EUW):

**Match ID:** EUW1_7351436977

**Game Mode:** CHERRY (Arena)

**Match Duration:** ~28 minutes 40 seconds

**Result:** Lost (Placement: 7th out of 8)

---

### Jovial Kar's Performance:
- **Champion:** Lee Sin
- **Role:** Support
- **Stats:**
  - Kills: 4
  - Deaths: 6
  - Assists: 4
  - KDA: 1.33
  - Damage to Champions: 28,630
  - Gold Earned: 9,500

**Items Built:** Shadow Flame, Spectre's Cowl, Guardian Angel, Ionian Boots of Lucidity

**Key Challenges Completed:**
- 34 Enemy Champion Immobilizations
- 160 Total Time CC Dealt
- 8 Takedowns total
- 79 Skill Shots Dodged

---

### Match Summary:
This was an Arena match (CHERRY game mode) where Jovial Kar played Lee Sin as a support. Despite a decent performance with good crowd control and multiple takedowns, the team placed 7th out of 8 teams, resulting in a loss. The match was relatively competitive with all players

 exit
